# Classificação de Subtipos de Células Sanguíneas - Mendeley Data
**COC891: Deep Learning 2026-2**

Este notebook implementa a comparação de **5 arquiteturas de Redes Neurais Convolucionais (CNNs)** para a classificação de subtipos de células sanguíneas utilizando validação cruzada (**K-Fold Cross-Validation**), data augmentation e transfer learning. O dataset de referência é o *"A dataset for microscopic peripheral blood cell images for development of automatic recognition systems"* (Acevedo et al., 2020), contendo 17.092 imagens originais distribuídas em 8 classes.

### Estrutura do Notebook:
1. Configurações Globais e Importações
2. Download e Organização do Dataset
3. Pipeline de Dados Eficiente (`tf.data.Dataset`)
4. Definição das 5 Topologias (Baseline, ResNet50, VGG16, InceptionV3, DenseNet121)
5. Loop de Validação Cruzada (K-Fold CV)
6. Experimento de Hiperparâmetros (Funções de Perda)
7. Consolidação de Resultados, Gráficos e Tabelas Comparativas

### 1. Configurações Globais e Importações

In [ ]:
import os
import zipfile
import shutil
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, InceptionV3, DenseNet121
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support

# Configurações globais
IMG_SIZE = (224, 224)  # Resolução padrão para transfer learning
BATCH_SIZE = 32
EPOCHS = 10
K_FOLDS = 5
RANDOM_STATE = 42
DATASET_DIR = './blood_cell_dataset'
RESULTS_DIR = './resultados_experimento'

os.makedirs(RESULTS_DIR, exist_ok=True)
print("TensorFlow versão:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices('GPU'))

### 2. Download e Organização do Dataset

Para rodar este notebook, você precisa das imagens do Mendeley Dataset:
1. **Download Manual (Recomendado):** Baixe o ZIP em https://data.mendeley.com/datasets/snkd93bnjr/1 (botão "Download All")
2. Coloque o arquivo `snkd93bnjr-1.zip` na mesma pasta deste notebook.
3. O código abaixo tentará descompactá-lo automaticamente ou baixá-lo via script se você não o fez.

In [ ]:
def check_and_extract_dataset():
    zip_path = './snkd93bnjr-1.zip'
    
    # Se a pasta do dataset já existe com imagens, não faz nada
    if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
        print("Dataset já descompactado em:", DATASET_DIR)
        return
        
    # Se o zip não estiver local, instrui o usuário ou tenta baixar de um espelho/link direto
    if not os.path.exists(zip_path):
        print("\n--- DATASET NÃO ENCONTRADO ---")
        print("Por favor, baixe o dataset manualmente em: https://data.mendeley.com/datasets/snkd93bnjr/1")
        print("Coloque o arquivo 'snkd93bnjr-1.zip' nesta mesma pasta e execute esta célula novamente.")
        print("Tentando fazer download automático (isso pode demorar vários minutos, tamanho ~3GB)...\n")
        
        # Link público direto para download do Mendeley Data
        url = "https://data.mendeley.com/public-files/datasets/snkd93bnjr/files/04bd99aa-869f-4318-97e3-0c464e815664/file_downloaded"
        try:
            urllib.request.urlretrieve(url, zip_path)
            print("Download concluído com sucesso!")
        except Exception as e:
            print("Falha no download automático:", e)
            print("Por favor, realize o download via navegador e coloque o arquivo zip na pasta local.")
            return
            
    print("Descompactando o dataset...")
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Extração concluída!")

check_and_extract_dataset()

### 3. Pipeline de Dados Eficiente (`tf.data.Dataset`)

Para não estourar a memória RAM do computador ou do Colab, utilizaremos o pipeline `tf.data` para carregar as imagens sob demanda. Mapearemos todas as imagens e suas respectivas labels.

In [ ]:
def build_filepath_dataframe():
    filepaths = []
    labels = []
    
    # O dataset extraído possui subpastas para cada tipo celular
    for root, dirs, files in os.walk(DATASET_DIR):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                path = os.path.join(root, file)
                label = os.path.basename(root)
                filepaths.append(path)
                labels.append(label)
                
    df = pd.DataFrame({'filepath': filepaths, 'label': labels})
    print(f"Total de imagens encontradas: {len(df)}")
    print("Distribuição de classes:")
    print(df['label'].value_counts())
    return df

df_data = build_filepath_dataframe()

# Mapeamento de texto para IDs numéricos
class_names = sorted(df_data['label'].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
df_data['label_idx'] = df_data['label'].map(class_to_idx)

print("Mapeamento de Classes:", class_to_idx)

In [ ]:
# Função auxiliar de carregamento e pré-processamento usando TensorFlow
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0  # Normalização
    return img, label

# Data Augmentation simples para generalização
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

def configure_dataset(df_fold, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((
        df_fold['filepath'].values,
        tf.keras.utils.to_categorical(df_fold['label_idx'].values, num_classes=len(class_names))
    ))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_training:
        ds = ds.shuffle(buffer_size=1000)
        ds = ds.batch(BATCH_SIZE)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE)
        
    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

### 4. Definição das 5 Topologias

Definiremos as 5 redes para comparação de desempenho. Usaremos Transfer Learning (pesos da ImageNet) para os modelos pré-treinados, treinando apenas a camada superior (*classifier head*) para acelerar o processo.

In [ ]:
num_classes = len(class_names)

# 1. Model Baseline CNN (Construída do Zero)
def get_baseline_model():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 2. VGG16 (Pre-trained)
def get_vgg16_model():
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False  # Congela pesos extratores
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 3. ResNet50 (Pre-trained)
def get_resnet50_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 4. InceptionV3 (Pre-trained)
def get_inceptionv3_model():
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 5. DenseNet121 (Pre-trained)
def get_densenet121_model():
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

### 5. Loop de Validação Cruzada (K-Fold CV)

Para garantir a robustez científica dos resultados comparativos (e atender à exigência do professor), executaremos a validação cruzada. Para economizar tempo computacional no Colab, aplicaremos a validação cruzada nos dois modelos principais (**Baseline** e **ResNet50**). Os outros modelos serão treinados e avaliados em um split fixo de teste (Hold-out).

In [ ]:
def evaluate_with_cross_validation(model_generator_fn, model_name):
    print(f"\n=== Iniciando {K_FOLDS}-Fold Cross-Validation para {model_name} ===")
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    fold_accuracies = []
    fold_histories = []
    
    # Itera sobre os folds
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_data['filepath'].values, df_data['label_idx'].values)):
        print(f"\n--- Treinando Fold {fold+1}/{K_FOLDS} ---")
        
        df_train = df_data.iloc[train_idx]
        df_val = df_data.iloc[val_idx]
        
        train_ds = configure_dataset(df_train, is_training=True)
        val_ds = configure_dataset(df_val, is_training=False)
        
        # Instancia e compila o modelo
        model = model_generator_fn()
        model.compile(
            optimizer=optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Callbacks
        early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        
        # Treinamento
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            callbacks=[early_stop],
            verbose=1
        )
        
        # Avaliação no fold
        val_loss, val_acc = model.evaluate(val_ds, verbose=0)
        print(f"Fold {fold+1} Acurácia de Validação: {val_acc:.4f}")
        
        fold_accuracies.append(val_acc)
        fold_histories.append(history.history)
        
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    print(f"\nResultado Final {model_name}: {mean_acc:.4f} (+/- {std_acc:.4f})")
    return fold_accuracies, fold_histories

# Executando a validação cruzada para os dois modelos foco
baseline_accuracies, baseline_histories = evaluate_with_cross_validation(get_baseline_model, "Baseline CNN")
resnet_accuracies, resnet_histories = evaluate_with_cross_validation(get_resnet50_model, "ResNet50")

In [ ]:
# Treinamento em Hold-Out (80% treino, 20% teste) para VGG, Inception e DenseNet para otimização de tempo
from sklearn.model_selection import train_test_split

df_train_ho, df_test_ho = train_test_split(
    df_data, test_size=0.20, stratify=df_data['label_idx'].values, random_state=RANDOM_STATE
)

train_ds_ho = configure_dataset(df_train_ho, is_training=True)
test_ds_ho = configure_dataset(df_test_ho, is_training=False)

ho_results = {}

def train_holdout(model_generator_fn, model_name):
    print(f"\n=== Treinando {model_name} em Hold-Out ===")
    model = model_generator_fn()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history = model.fit(
        train_ds_ho,
        validation_data=test_ds_ho,
        epochs=EPOCHS,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Predição para relatórios de classificação
    y_true = []
    y_pred = []
    for x, y in test_ds_ho:
        preds = model.predict(x, verbose=0)
        y_true.extend(np.argmax(y.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)
    
    ho_results[model_name] = {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'history': history.history,
        'cm': cm
    }
    print(f"{model_name} Acurácia final no Teste: {acc:.4f}")

train_holdout(get_vgg16_model, "VGG16")
train_holdout(get_inceptionv3_model, "InceptionV3")
train_holdout(get_densenet121_model, "DenseNet121")

### 6. Experimento de Hiperparâmetros (Funções de Perda)

Como sugerido no arquivo de comentários do professor, variaremos as funções de perda para comparar seu impacto no aprendizado. Utilizaremos o modelo **Baseline CNN** e compararemos a clássica `Categorical Crossentropy` com a `Kullback-Leibler Divergence (KL Divergence)`.

In [ ]:
loss_functions = {
    'Categorical Crossentropy': 'categorical_crossentropy',
    'KL Divergence': 'kl_divergence'
}

loss_experiment_results = {}

for loss_name, loss_fn in loss_functions.items():
    print(f"\n=== Treinando Baseline CNN com função de perda: {loss_name} ===")
    model = get_baseline_model()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss=loss_fn,
        metrics=['accuracy']
    )
    
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history = model.fit(
        train_ds_ho,
        validation_data=test_ds_ho,
        epochs=EPOCHS,
        callbacks=[early_stop],
        verbose=1
    )
    
    val_loss, val_acc = model.evaluate(test_ds_ho, verbose=0)
    loss_experiment_results[loss_name] = {
        'val_accuracy': val_acc,
        'history': history.history
    }
    print(f"Acurácia de Validação com {loss_name}: {val_acc:.4f}")

### 7. Consolidação de Resultados, Gráficos e Tabelas Comparativas

Geração e salvamento das curvas de perda, acurácia e matriz de confusão para o relatório técnico.

In [ ]:
# 1. Plotando a comparação das funções de perda
plt.figure(figsize=(12, 5))
for loss_name, res in loss_experiment_results.items():
    plt.plot(res['history']['val_loss'], label=f'Val Loss ({loss_name})')
plt.title('Impacto da Função de Perda na Convergência (Val Loss)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
loss_plot_path = os.path.join(RESULTS_DIR, 'experimento_loss_functions.png')
plt.savefig(loss_plot_path)
plt.show()
print("Gráfico salvo em:", loss_plot_path)

In [ ]:
# 2. Plotando curvas de aprendizado dos modelos Holdout (VGG, Inception, DenseNet)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (model_name, res) in enumerate(ho_results.items()):
    axes[idx].plot(res['history']['accuracy'], label='Treino')
    axes[idx].plot(res['history']['val_accuracy'], label='Validação')
    axes[idx].set_title(f'Acurácia - {model_name}')
    axes[idx].set_xlabel('Épocas')
    axes[idx].set_ylabel('Acurácia')
    axes[idx].legend()
    axes[idx].grid(True)
ho_plots_path = os.path.join(RESULTS_DIR, 'curvas_aprendizado_modelos.png')
plt.savefig(ho_plots_path)
plt.show()
print("Gráficos de acurácia salvos em:", ho_plots_path)

In [ ]:
# 3. Plotando as matrizes de confusão para os modelos Holdout
for model_name, res in ho_results.items():
    plt.figure(figsize=(8, 6))
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Matriz de Confusão - {model_name}')
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, f'matriz_confusao_{model_name}.png')
    plt.savefig(cm_path)
    plt.show()
    print(f"Matriz de confusão para {model_name} salva em:", cm_path)

In [ ]:
# 4. Construção da Tabela Comparativa de Métricas Finais
final_metrics = []

# Adiciona Baseline e ResNet50 (Média da validação cruzada)
final_metrics.append({
    'Modelo': 'Baseline CNN (K-Fold CV)',
    'Acurácia': f"{np.mean(baseline_accuracies):.4f} (+/- {np.std(baseline_accuracies):.4f})",
    'Precision (weighted)': 'N/A (CV)',
    'Recall (weighted)': 'N/A (CV)',
    'F1-Score (weighted)': 'N/A (CV)'
})
final_metrics.append({
    'Modelo': 'ResNet50 (K-Fold CV)',
    'Acurácia': f"{np.mean(resnet_accuracies):.4f} (+/- {np.std(resnet_accuracies):.4f})",
    'Precision (weighted)': 'N/A (CV)',
    'Recall (weighted)': 'N/A (CV)',
    'F1-Score (weighted)': 'N/A (CV)'
})

# Adiciona os modelos holdout
for model_name, res in ho_results.items():
    final_metrics.append({
        'Modelo': f"{model_name} (Hold-out 80/20)",
        'Acurácia': f"{res['accuracy']:.4f}",
        'Precision (weighted)': f"{res['precision']:.4f}",
        'Recall (weighted)': f"{res['recall']:.4f}",
        'F1-Score (weighted)': f"{res['f1']:.4f}"
    })

df_metrics = pd.DataFrame(final_metrics)
df_metrics.to_csv(os.path.join(RESULTS_DIR, 'tabela_comparativa_modelos.csv'), index=False)
df_metrics